# ML Pipeline — Exercises

**Companion to deck 05 (5-step mantra).** Build a Titanic classifier end-to-end. Same shape as every camp baseline.

<a href="https://colab.research.google.com/github/Petkub/MachineLearningLab/blob/main/colab_exercises/05_ml_pipeline.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

df = sns.load_dataset('titanic').rename(columns={
    'pclass': 'Pclass', 'sex': 'Sex', 'age': 'Age',
    'fare': 'Fare', 'survived': 'Survived',
})
# encode + impute
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df['Age'] = df['Age'].fillna(df['Age'].mean())
df = df.dropna(subset=['Fare'])
print(df.shape)
df.head()

---
## Problem 01 — separate X and y

Tasks:
1. Build `X` — features `['Pclass', 'Sex', 'Age', 'Fare']` from `df`.
2. Build `y` — the `Survived` column.
3. Print shapes.

In [ ]:
# TODO
X = ...
y = ...

print('X:', X.shape, '| y:', y.shape)

In [ ]:
assert X.shape[1] == 4, f'X should have 4 columns, has {X.shape[1]}'
assert 'Survived' not in X.columns, 'target leaked into X — remove it'
assert len(X) == len(y), 'X and y have different row counts'
print('Q1 ok')

<details><summary>Hint</summary>

Pick a subset of columns by passing a **list of names** (note the double brackets — outer is indexing, inner is the list). Pull a single column out as a Series with single brackets and the column name. **Trap**: never include the target column in `X` — that's a data leak; the model would just copy it.

</details>

---
## Problem 02 — split TWO ways and compare

A naive 80/20 split sometimes returns a test set with a wildly different class balance from the train set — pure luck. Stratifying fixes that. Prove it.

Tasks:
1. Build `(_, _, y_train_a, y_test_a)` from a normal split (`random_state=0`, no stratify).
2. Build `(_, _, y_train_b, y_test_b)` from a stratified split (`random_state=0`, `stratify=y`).
3. Save the **drift** in survival rate `drift_a = abs(y_train_a.mean() - y_test_a.mean())` and `drift_b` for the stratified split.

Stratified should drift much less than random. **Decide:** by how many percentage points?

In [ ]:
# TODO
# split A — random
_, _, y_train_a, y_test_a = ...

# split B — stratified
_, _, y_train_b, y_test_b = ...

drift_a = abs(y_train_a.mean() - y_test_a.mean())
drift_b = abs(y_train_b.mean() - y_test_b.mean())

print(f'random drift:     {drift_a:.4f}')
print(f'stratified drift: {drift_b:.4f}')
print(f'stratify shrunk drift by {drift_a - drift_b:.4f}')

In [ ]:
assert drift_b < drift_a, 'stratify should drift less than random — re-check stratify=y'
assert drift_b < 0.005, f'stratified drift should be tiny, got {drift_b}'
print('Q2 ok — stratify keeps test class ratio honest')

<details><summary>Hint</summary>

Call `train_test_split` twice with the same data and same `random_state`, only the `stratify=y` keyword changes between calls. Stratify forces the same class proportion in train and test (within rounding). Random splits can drift a few percentage points by luck.

</details>

---
## Problem 03 — fit + predict + evaluate

Train a `LogisticRegression` (use `max_iter=1000` so it converges). Predict on the test set. Print accuracy.

In [ ]:
# TODO
model = ...
# fit on train
# predict on test
preds = ...
acc = ...

print('accuracy:', round(acc, 3))

In [ ]:
assert 0.70 < acc < 0.90, f'accuracy out of expected range: {acc}'
assert len(preds) == len(y_test), 'preds length mismatch'
print('Q3 ok — LogReg accuracy:', round(acc, 3))

<details><summary>Hint</summary>

Three universal sklearn calls:
1. **Construct** the model — pass `max_iter=1000` so LogReg has enough optimizer steps.
2. `.fit(X_train, y_train)` — train.
3. `.predict(X_test)` — get an array of predicted labels.

Then `accuracy_score(y_test, preds)` returns the fraction correct. Same shape works for *every* sklearn classifier.

</details>

---
## Problem 04 — swap models without changing the pipeline

Train and score three models: LogReg, RandomForest, KNN. Save accuracies in a dict `scores`.

**Goal:** prove to yourself that only one line changes per model.

In [ ]:
# TODO
scores = {}

# for each model: fit on train, predict on test, record accuracy in scores
# use these:
#   LogisticRegression(max_iter=1000)
#   RandomForestClassifier(n_estimators=100, random_state=42)
#   KNeighborsClassifier(n_neighbors=5)

for name, score in scores.items():
    print(f'{name:25s} {score:.3f}')

In [ ]:
assert set(scores.keys()) >= {'logreg', 'rf', 'knn'}, 'scores dict must have keys: logreg, rf, knn'
for name, s in scores.items():
    assert 0.6 < s < 0.95, f'{name} score out of range: {s}'
print('Q4 ok — winner:', max(scores, key=scores.get))

<details><summary>Hint</summary>

Build a **dict of name → model instance**, then a single loop over `.items()`. Inside the loop: fit, predict, store the accuracy under the name key in `scores`. Same train/test data for every model — that's the whole point. If you find yourself writing three near-identical blocks, you're doing it wrong.

</details>

---
## Problem 05 — beyond accuracy

Use the best model from Q4. Compute confusion matrix, precision, recall, F1.

Question to answer in your head: which mistake type is more common — predicting *survived* when they didn't (FP), or *died* when they did (FN)?

In [ ]:
# TODO
best = ...   # winning model from Q4 — refit if needed, or reuse preds
best_preds = ...

cm = confusion_matrix(y_test, best_preds)
print('confusion matrix:\n', cm)
print('precision:', round(precision_score(y_test, best_preds), 3))
print('recall:   ', round(recall_score(y_test, best_preds), 3))
print('f1:       ', round(f1_score(y_test, best_preds), 3))

In [ ]:
assert cm.shape == (2, 2), 'confusion matrix should be 2x2 for binary'
tn, fp, fn, tp = cm.ravel()
print(f'TN={tn} FP={fp} FN={fn} TP={tp}')
print('Q5 ok')

<details><summary>Hint</summary>

`scores` is a dict mapping name → accuracy. To pick the winner, use `max(scores, key=scores.get)` — returns the **key** with the largest value. Refit that model on `X_train` (or reuse it if you stored it earlier), then `.predict(X_test)`. Pass the predictions through the four metric functions you imported. **Question to ponder**: a high false-negative count (FN) means we're missing real survivors — depending on the use case, that might be worse than false alarms.

</details>

---
## Problem 06 — accuracy trap

Build a baseline model that **always predicts the majority class** (everyone died). Compute its accuracy.

If your fancy model only beats this by 5 points, your fancy model isn't fancy.

In [ ]:
# TODO — no sklearn needed, no numpy needed
majority = ...        # most common class in y_train (0 or 1)
dummy_preds = ...     # a list of length len(y_test), every entry = majority
dummy_acc = ...       # fraction correct vs y_test

print('majority class:', majority)
print('dummy accuracy:', round(dummy_acc, 3))
print('your best model:', round(max(scores.values()), 3))
print('lift over dummy:', round(max(scores.values()) - dummy_acc, 3))

In [ ]:
assert majority in (0, 1)
assert len(dummy_preds) == len(y_test)
assert max(scores.values()) > dummy_acc, 'your model should beat the dummy'
print('Q6 ok — your lift:', round(max(scores.values()) - dummy_acc, 3))

<details><summary>Hint</summary>

Most common value in `y_train`: pandas Series have a `.mode()` method that returns a Series — take the first element with `[0]`. Build a constant list with `[majority] * len(y_test)`. Then score it the same way you've scored real models — `accuracy_score(y_test, dummy_preds)`.

</details>

---
## Problem 07 — find the bugs in this pipeline

The cell below trains a model and reports an accuracy of `0.99`. That's suspicious. Find and fix **three bugs**.

Things to look for: data leak, wrong split direction, wrong evaluation set.

In [ ]:
# BROKEN — fix three things, then save the corrected accuracy in `acc_q7`.
#
# Bugs to find:
#  - the target column is hiding inside X
#  - the test/train order is wrong (split returns 4 things in a fixed order)
#  - the model is evaluated on the wrong set, which is why accuracy looks too good

X_bad = df[['Pclass', 'Sex', 'Age', 'Fare', 'Survived']]
y_bad = df['Survived']

X_test_b, X_train_b, y_test_b, y_train_b = train_test_split(
    X_bad, y_bad, test_size=0.2, random_state=42, stratify=y_bad)

m = LogisticRegression(max_iter=1000)
m.fit(X_train_b, y_train_b)
acc_q7 = accuracy_score(y_train_b, m.predict(X_train_b))

print('reported accuracy:', round(acc_q7, 3))

In [ ]:
assert acc_q7 < 0.95, f'still leaky/wrong — got {acc_q7}, real LogReg should be ~0.78–0.85'
assert acc_q7 > 0.65, f'too low — over-fixed something: {acc_q7}'
print('Q7 ok — honest accuracy:', round(acc_q7, 3))

<details><summary>Hint</summary>

Three independent failures:
- A feature matrix that contains the target column gives the model a free pass — it just copies the answer. Drop the target from `X`.
- `train_test_split` returns `X_train, X_test, y_train, y_test` in that order. The current code unpacks them backwards.
- After fitting on the train set, evaluating again on the train set tells you nothing. Score on the **held-out** test set.

</details>

---
## Solutions

<details><summary>Show all solutions</summary>

```python
# Q1
X = df[['Pclass', 'Sex', 'Age', 'Fare']]
y = df['Survived']

# Q2 — random vs stratified
_, _, y_train_a, y_test_a = train_test_split(X, y, test_size=0.2, random_state=0)
_, _, y_train_b, y_test_b = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
drift_a = abs(y_train_a.mean() - y_test_a.mean())
drift_b = abs(y_train_b.mean() - y_test_b.mean())

# (re-run the proper split for Q3+ work)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Q3
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)

# Q4
candidates = {
    'logreg': LogisticRegression(max_iter=1000),
    'rf':     RandomForestClassifier(n_estimators=100, random_state=42),
    'knn':    KNeighborsClassifier(n_neighbors=5),
}
scores = {}
for name, m in candidates.items():
    m.fit(X_train, y_train)
    scores[name] = accuracy_score(y_test, m.predict(X_test))

# Q5
best_name = max(scores, key=scores.get)
best = candidates[best_name]
best_preds = best.predict(X_test)

# Q6
majority = y_train.mode()[0]
dummy_preds = [majority] * len(y_test)
dummy_acc = accuracy_score(y_test, dummy_preds)

# Q7 — fixes
#   - drop 'Survived' from X
#   - split returns X_train, X_test, y_train, y_test (in that order)
#   - score on test, not train
X_clean = df[['Pclass', 'Sex', 'Age', 'Fare']]
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clean, y, test_size=0.2, random_state=42, stratify=y)
m = LogisticRegression(max_iter=1000).fit(X_train_c, y_train_c)
acc_q7 = accuracy_score(y_test_c, m.predict(X_test_c))
```
</details>